# D2.4 · Containment at machine speed

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.3 · Scoping an agentic incident](https://spbreed.github.io/cyber-commons/lessons/D2.3.html)**.

| | |
|---|---|
| Tools used | agentgateway, Keycloak |

## What this lesson is

**What it covers.** Exercise the ladder against a live misbehaving agent.

**Why a security engineer needs it.** Mass revocation takes down the business. The control it builds is: throttle → scope-reduce → reroute → force HITL → revoke → hard stop, in order.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You have to stop it faster than it acts. That means the containment path — revoke, cut the gateway, kill the loop — is a thing built in advance, because improvising it takes longer than the incident does.

> **At CyberTravels.** You have to stop CyberTravels faster than it issues refunds. The containment path is something CyberTravels builds in advance, because improvising it takes longer than the incident.

## 2 · The framework

```
   containment paths, in order of how fast they actually work

   1  revoke the credential      seconds, if it is short-lived
   2  cut it at the gateway      seconds, if there is a gateway
   3  kill the loop              minutes, if you know where it runs
   4  disable the integration    hours

   built in advance. improvised, path 1 takes longer than the incident.
```

Containment has always been a race. With an agent, the other runner got much
faster and you did not.

The numbers decide the design. An agent operating at 300 actions per minute
completes 2,400 further actions during an eight-minute approval cycle, against
about 60 under automated containment. That ratio is the argument for
pre-authorised, automated revocation of non-human identities.

The asymmetry that makes it safe: revoking a **human's** access needs care,
because a false positive locks a person out mid-shift. Revoking a **non-human**
identity is cheap to get wrong — the agent re-requests, or an on-call re-enables
it in a minute. So the two should have different policies, and almost nowhere do.

## 3 · The procedure, as a skill

Eight minutes of approval is 2,400 actions. The skill races the rate against the delay, times the whole containment path — of which the revocation itself is twelve seconds — and classifies which signals may auto-revoke against a non-human subject.

### The skill — [`skills/response/machine-speed-containment/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/machine-speed-containment/SKILL.md)

```yaml
name: machine-speed-containment
description: >-
  Race an agent's action rate against the approval delay in front of
  containment, time the whole containment path end to end, and decide which
  signals may auto-revoke. Use when containment requires a human and the subject
  acts hundreds of times a minute.
allowed-tools: Read, Grep, Glob
```

# Eight minutes of approval is 2,400 actions

Containment that waits for a person is measured against a subject that does not
wait. The arithmetic is not close, and it is the argument for pre-authorising
revocation for a named set of signals — with the false-revocation cost stated,
because that is the objection.

## When to use this

Designing containment for agent workloads, and after any incident where the
containment step was correct and late.

## Procedure

**1 — Measure the subject's rate.** Actions per minute, observed rather than
designed. Multiply by the approval delay to get the actions taken while
somebody decides.

**2 — Time the whole path, not the revocation.** Detection, triage, decision,
approval, execution, propagation. The revocation itself is usually seconds and
the path is usually minutes; reporting only the last step makes the problem
invisible.

**3 — Find the dominant term.** It is almost always human approval or
propagation delay, and it is almost never the API call. Optimise the dominant
term or nothing changes.

**4 — Classify signals for auto-revocation.** For each, its precision and what a
false revocation costs. High-precision signals against a non-human subject are
the candidates: revoking an agent's token wrongly costs a restarted run.

**5 — Set the policy asymmetrically.** Auto-revoke agent credentials on
high-precision signals; keep a human in front of anything that affects a person's
access. State both halves so the policy survives review.

## Example

**Input** — the fixture committed at the top of [`scripts/machine_speed_containment.py`](scripts/machine_speed_containment.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
   agent rate   human 8min   auto 12s   ratio
----------------------------------------------
       30/min          240          6    40.0×
      120/min          960         24    40.0×
      300/min         2400         60    40.0×
     1200/min         9600        240    40.0×

At 300/min an 8-minute approval costs 2,400 further actions.
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "race": {"actions_per_min": 0, "approval_minutes": 0, "actions_during_approval": 0},
  "path": [{"step": "str", "seconds": 0}],
  "total_seconds": 0,
  "dominant_step": "str",
  "signals": [{"name": "str", "precision": 0.0, "subject": "agent|human",
               "auto_revoke": false, "false_revocation_cost": "str"}]
}
```

## Failure modes

- **Timing the revocation.** It is the fast part.
- **One policy for agents and people.** The costs differ by orders of
  magnitude.
- **Auto-revoking on a low-precision signal.** Precision is the entry
  requirement.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/machine-speed-containment/scripts/machine_speed_containment.py
SCRIPT = "skills/response/machine-speed-containment/scripts/machine_speed_containment.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The race table shows 2,400 versus 60 actions at 300/min for an eight-minute approval. The full containment path totals about 920 seconds, of which the revocation itself is 12. Four of five signals auto-revoke for non-human identities and none do for a human subject, cutting the path to 21 seconds and preventing roughly 4,500 actions.

## Your turn

Time your own containment path end to end, step by step. The revocation is almost never the slow part — queue depth and approval are, and both are policy choices rather than technical limits.

---

**Next → [D2.5 · Replay and forensics](https://spbreed.github.io/cyber-commons/lessons/D2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*